# Mantenimiento Predictivo — KAESER CSDX165

**Proyecto Final · Curso de Inteligencia Artificial Industrial**  
**Autor:** Felipe Soto

---

### Problema
Predecir la advertencia de diferencial de presión en filtros de aceite (**0011 W**)  
y filtros de aire (**0013 W**) antes de que ocurra, a partir de datos sensoriales  
registrados a 1 Hz por el controlador SIGMA CONTROL 2 (archivos `.dat`).

### Estado del Notebook

| Commit | Secciones | Estado |
|--------|-----------|--------|
| **C1** | Imports · Configuración · Carga DAT · Parser de eventos | ✅ *este* |
| C2 | Extracción de canales · Features · Labels | 🔲 |
| C3 | Ensamblado del dataset | 🔲 |
| C4 | Modelos (DT · RF · MLP) | 🔲 |
| C5 | Evaluación · Interpretación · Conclusiones | 🔲 |

In [ ]:
# ── 0.1  Imports ──────────────────────────────────────────────────────────────
import os, re, glob, struct
from datetime import datetime

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt   # usado desde C2 en adelante

# Semilla de reproducibilidad — se fija aquí, nunca se sobreescribe
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Imports OK')

---
## 0.2  Configuración

Todas las rutas, constantes y esquemas de columnas se definen aquí.  
El resto del notebook **no repite** estos valores.

In [ ]:
# ── Rutas base ────────────────────────────────────────────────────────────────
ROOT      = r'C:\Users\Felipe\Desktop\Industrias\2026\IA\proyecto\equipos'
PROJ_ROOT = r'C:\Users\Felipe\Desktop\Industrias\2026\IA\proyecto'

# Fragmentos de ruta intermedios (mantienen COMPRESSORS legible)
_d1  = os.path.join(ROOT, 'Compresor CSDX165 Aire Industrial N\u00ba1', 'CSDX165_INDN\u00ba1')
_d2  = os.path.join(ROOT, 'Compresor CSDX165 Aire Industrial N\u00ba2', 'CSDX165_INDN\u00ba2')
_bf1 = 'BF_101803.1_1171_2026-05-27T09.48.42-04.00'
_bf2 = 'BF_101803.2_1001_2026-05-27T09.54.00-04.00'

COMPRESSORS = {
    'INDN\u00ba1': {
        'dat_root':    os.path.join(_d1, 'datarecorder'),
        'log_path':    os.path.join(_d1, _bf1, 'reports_lang', 'CompressorMsgs.txt'),
        'target_code': '0011',
        'target_cat':  'W',
        'dat_start':   datetime(2026, 3, 18),
        'dat_end':     datetime(2026, 5, 27),
    },
    'INDN\u00ba2': {
        'dat_root':    os.path.join(_d2, 'datarecorder'),
        'log_path':    os.path.join(_d2, _bf2, 'reports_lang', 'CompressorMsgs.txt'),
        'target_code': '0013',
        'target_cat':  'W',
        'dat_start':   datetime(2025, 6, 23),
        'dat_end':     datetime(2026, 5, 27),
    },
}
print('Paths OK')

In [ ]:
# ── Formato binario DAT ───────────────────────────────────────────────────────
DAT_VALID_SIZE    = 954_664   # bytes — archivos de otro tamaño se descartan
DAT_DATA_OFFSET   = 4264      # inicio del bloque de datos (tras la cabecera)
DAT_REC_SIZE      = 264       # bytes por registro
DAT_N_RECORDS     = 3600      # registros por archivo (1 Hz x 3600 s = 1 h)
DAT_N_CHANNELS    = 130       # canales int16 por registro
DAT_MISSING_INT16 = -32768    # valor centinela -> NaN

# ── Canales seleccionados (Tier-1 + Tier-2) ───────────────────────────────────
# Confirmados por auditoría de canal (_channel_map.py / _channel_live_check.py)
CH = {
    'cs':         0,   # Compressor status    — bitmask (en marcha/cargado)
    'pressure':   4,   # System pressure      — x0.01 -> bar
    'oil_temp':   6,   # Oil separator temp.  — x0.01 -> grados C
    'inlet_temp': 9,   # Inlet temperature    — x0.1  -> grados C  (proxy ambiental)
    'speed_sp':  12,   # Compressor speed SP  — x10   -> RPM       (proxy de carga)
    # Tier-2
    'fan_speed': 38,   # Oil cooler fan speed — x10   -> RPM
    'etm_valve': 44,   # ETM valve pos. SP    — x0.1  -> %
    'etm_fan_sp':45,   # ETM fan speed SP     — x0.1  -> %
    'etm_ctrl':  47,   # ETM dT ctrl. var.    — x0.1  -> %
}

CH_SCALE = {
    'cs': 1.0,   'pressure': 0.01, 'oil_temp': 0.01,
    'inlet_temp': 0.1, 'speed_sp': 10.0, 'fan_speed': 10.0,
    'etm_valve': 0.1,  'etm_fan_sp': 0.1, 'etm_ctrl': 0.1,
}

CH_UNIT = {
    'cs': 'bitmask', 'pressure': 'bar',  'oil_temp': 'C',
    'inlet_temp': 'C', 'speed_sp': 'RPM', 'fan_speed': 'RPM',
    'etm_valve': '%',  'etm_fan_sp': '%', 'etm_ctrl': '%',
}

# Orden de extraccion — define el orden de columnas en los arrays raw
CH_NAMES        = list(CH.keys())             # ['cs', 'pressure', ...]
EXTRACT_INDICES = [CH[k] for k in CH_NAMES]   # [0, 4, 6, 9, 12, 38, 44, 45, 47]

# ── Constantes de etiquetado (usadas desde la Seccion 2) ─────────────────────
PREDICTION_HORIZON_H = 24   # horas previas al onset = etiqueta positiva
RECOVERY_EXCLUSION_H =  4   # horas post-despeje excluidas del entrenamiento
EPISODE_GAP_DAYS     =  7   # brecha minima (dias) entre episodios distintos

ZONE_NEGATIVE = 'negative'
ZONE_PRE_WARN = 'pre_warning'
ZONE_ACTIVE   = 'active_warning'
ZONE_RECOVERY = 'recovery'
TARGET_COL    = 'label_extended'

# ── Columnas del modelo (forward declaration — pobladas en la Seccion 2) ─────
FEATURE_COLS = [
    'p_mean_1h',        'p_std_1h',          'p_mean_24h',        'p_trend_24h',
    'oil_temp_mean_1h', 'oil_temp_max_1h',   'oil_temp_mean_24h', 'oil_temp_trend_24h',
    'inlet_temp_mean_24h',
    'speed_mean_1h',    'speed_mean_24h',    'speed_trend_24h',
    'load_frac_1h',     'load_frac_24h',     'starts_24h',
    'fan_speed_mean_1h','etm_valve_mean_1h', 'etm_fan_sp_mean_1h',
    'month_sin',        'month_cos',         'hour_sin',          'hour_cos',
    'hours_loaded_since_clear',
]  # 23 features

print(f'Config OK  —  {len(FEATURE_COLS)} features declaradas,  {len(CH)} canales seleccionados')

---
## 1  Utilidades de Carga DAT

Dos funciones de bajo nivel. Sin lógica de features ni escalado.

| Función | Entrada | Salida |
|---------|---------|--------|
| `list_dat_files(dat_root)` | directorio del recorder | lista ordenada de rutas `.dat` válidas |
| `read_dat_hour(filepath)` | ruta de un `.dat` | `(hour_utc, raw_array)` |

In [ ]:
# ── 1.1  list_dat_files ───────────────────────────────────────────────────────
def list_dat_files(dat_root):
    """
    Return sorted list of valid DAT file paths under dat_root.
    Rejects files whose size differs from DAT_VALID_SIZE (truncated/partial).
    """
    files = sorted(
        glob.glob(os.path.join(dat_root, '**', 'mcs_*.dat'), recursive=True)
    )
    return [f for f in files if os.path.getsize(f) == DAT_VALID_SIZE]


# ── 1.2  read_dat_hour ────────────────────────────────────────────────────────
def read_dat_hour(filepath):
    """
    Read one DAT file.

    Returns
    -------
    hour_utc : datetime
        UTC time of record 0 (= file-hour start).
    raw : ndarray, shape (3600, len(EXTRACT_INDICES)), float64
        Raw int16 channel values; sentinel (-32768) replaced with NaN.
        Column order: CH_NAMES  /  EXTRACT_INDICES.
        Scale factors are NOT applied here.
    """
    with open(filepath, 'rb') as fh:
        data = fh.read()

    # UTC timestamp from first record (uint32 LE at DATA_OFFSET)
    ts0      = struct.unpack_from('<I', data, DAT_DATA_OFFSET)[0]
    hour_utc = datetime.utcfromtimestamp(ts0)

    # Read every record as int16 words.
    # Layout per 264-byte record: [ts_lo, ts_hi, ch0, ch1, ..., ch129]
    #   264 bytes / 2 = 132 int16 per record
    #   cols 0-1  : timestamp (two int16 = one uint32)
    #   cols 2-131: 130 channel values
    flat = np.frombuffer(
        data, dtype='<i2',
        offset=DAT_DATA_OFFSET,
        count=DAT_N_RECORDS * (DAT_REC_SIZE // 2)
    ).reshape(DAT_N_RECORDS, DAT_REC_SIZE // 2)   # (3600, 132)

    ch_all = flat[:, 2:].astype(np.float64)        # (3600, 130)
    ch_all[ch_all == DAT_MISSING_INT16] = np.nan

    return hour_utc, ch_all[:, EXTRACT_INDICES]


print('DAT utilities OK')

---
## 2  Utilidades de Carga del Log de Eventos

| Función | Entrada | Salida |
|---------|---------|--------|
| `parse_event_log(log_path, code, cat)` | ruta + filtro de código/categoría | `df_events` |
| `build_episode_table(df_events, start, end)` | `df_events` + ventana DAT | `df_episodes` |

In [ ]:
# ── 2.1  parse_event_log ──────────────────────────────────────────────────────
_HEADER_RE = re.compile(
    r'^(\d{4})\s+([OWA])\s+([cga])\s+'
    r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})([-+]\d{2}):?(\d{2})'
)

def parse_event_log(log_path, target_code, target_cat):
    """
    Parse CompressorMsgs.txt and return a DataFrame of target events.

    Parameters
    ----------
    log_path    : str  — path to CompressorMsgs.txt
    target_code : str  — 4-digit event code, e.g. '0011'
    target_cat  : str  — category char: 'O', 'W', or 'A'

    Returns
    -------
    DataFrame columns:
        timestamp   datetime  — local time as recorded (no UTC conversion)
        state       str       — 'c' (onset) | 'g' (clear)
        tz_offset_h int       — signed hours offset from log line
    Sorted ascending by timestamp.
    Exact (timestamp, state) duplicates removed.
    """
    rows = []
    with open(log_path, encoding='utf-8', errors='replace') as fh:
        for line in fh:
            m = _HEADER_RE.match(line.strip())
            if not m:
                continue
            if m.group(1) != target_code or m.group(2) != target_cat:
                continue
            dt   = datetime.strptime(m.group(4), '%Y-%m-%dT%H:%M:%S')
            tz_h = int(m.group(5))
            rows.append({'timestamp': dt, 'state': m.group(3), 'tz_offset_h': tz_h})

    if not rows:
        return pd.DataFrame(columns=['timestamp', 'state', 'tz_offset_h'])

    return (
        pd.DataFrame(rows)
          .sort_values('timestamp')
          .drop_duplicates(subset=['timestamp', 'state'])
          .reset_index(drop=True)
    )


print('parse_event_log OK')

In [ ]:
# ── 2.2  build_episode_table ──────────────────────────────────────────────────
def build_episode_table(df_events, dat_start, dat_end):
    """
    Group events into maintenance episodes (EPISODE_GAP_DAYS threshold).
    Returns only episodes whose first onset falls inside [dat_start, dat_end].

    Parameters
    ----------
    df_events : DataFrame from parse_event_log()
    dat_start : datetime — first date of DAT coverage
    dat_end   : datetime — last date of DAT coverage

    Returns
    -------
    DataFrame columns:
        episode_id    int
        first_onset   datetime
        last_clear    datetime  (NaT if episode never cleared in log)
        n_onsets      int
        in_dat        bool
        duration_days float     (NaN if last_clear is NaT)
    Only in_dat == True rows returned.
    """
    onsets = df_events[df_events['state'] == 'c'].reset_index(drop=True)
    clears = df_events[df_events['state'] == 'g'].reset_index(drop=True)

    _empty = pd.DataFrame(
        columns=['episode_id', 'first_onset', 'last_clear',
                 'n_onsets', 'in_dat', 'duration_days']
    )
    if onsets.empty:
        return _empty

    # ── Cluster onsets by EPISODE_GAP_DAYS gap ────────────────────────────────
    ep_groups = []
    current   = [onsets.iloc[0]['timestamp']]
    for i in range(1, len(onsets)):
        gap_s = (
            onsets.iloc[i]['timestamp'] - onsets.iloc[i - 1]['timestamp']
        ).total_seconds()
        if gap_s > EPISODE_GAP_DAYS * 86_400:
            ep_groups.append(current)
            current = []
        current.append(onsets.iloc[i]['timestamp'])
    ep_groups.append(current)

    # ── Build one row per episode ─────────────────────────────────────────────
    rows = []
    for ep_id, ep_ts in enumerate(ep_groups, 1):
        first_onset = ep_ts[0]
        # Last clear: any 'g' after this episode's first onset
        #             and before the next episode's first onset
        upper  = ep_groups[ep_id][0] if ep_id < len(ep_groups) else pd.Timestamp.max
        mask   = (clears['timestamp'] > first_onset) & (clears['timestamp'] < upper)
        last_clear = clears.loc[mask, 'timestamp'].max() if mask.any() else pd.NaT
        dur = (
            (last_clear - first_onset).total_seconds() / 86_400
            if pd.notna(last_clear) else np.nan
        )
        rows.append({
            'episode_id':    ep_id,
            'first_onset':   first_onset,
            'last_clear':    last_clear,
            'n_onsets':      len(ep_ts),
            'in_dat':        dat_start <= first_onset <= dat_end,
            'duration_days': round(dur, 2) if pd.notna(last_clear) else np.nan,
        })

    df_eps = pd.DataFrame(rows)
    return df_eps[df_eps['in_dat']].reset_index(drop=True)


print('build_episode_table OK')

---
## Verificación — Commit 1

Comprueba que las utilidades de carga producen los conteos y marcas de tiempo  
esperados, confirmados en la auditoría previa.

| Compressor | DAT files | DAT start | DAT end | Onsets in-DAT | Episodios in-DAT |
|------------|-----------|-----------|---------|---------------|------------------|
| INDNº1 | ~1 650 | 2026-03-18 | 2026-05-27 | 17 | 1 |
| INDNº2 | ~8 045 | 2025-06-23 | 2026-05-27 | 19 | 4 |

In [ ]:
SEP  = '=' * 70
DSEP = '-' * 70

print(SEP)
print('  COMMIT 1  -  VERIFICACION')
print(SEP)

for comp_id, cfg in COMPRESSORS.items():
    code = cfg['target_code']
    cat  = cfg['target_cat']

    print(f'\n{DSEP}')
    print(f'  Compressor   : {comp_id}   (evento objetivo: {code} {cat})')
    print(DSEP)

    # ── DAT files ─────────────────────────────────────────────────────────────
    files = list_dat_files(cfg['dat_root'])
    print(f'  DAT files    : {len(files):,}')

    if files:
        t_first, _ = read_dat_hour(files[0])
        t_last,  _ = read_dat_hour(files[-1])
        print(f'  DAT first    : {t_first}  UTC')
        print(f'  DAT last     : {t_last}  UTC')
    else:
        print('  ADVERTENCIA  : no se encontraron archivos DAT validos.')

    # ── Event log ─────────────────────────────────────────────────────────────
    df_ev   = parse_event_log(cfg['log_path'], code, cat)
    n_came  = (df_ev['state'] == 'c').sum()
    n_gone  = (df_ev['state'] == 'g').sum()
    print(f'\n  Eventos {code} {cat} : {len(df_ev):,} total'
          f'  ({n_came} onsets  /  {n_gone} clearances)')

    # ── Episodes ──────────────────────────────────────────────────────────────
    df_ep = build_episode_table(df_ev, cfg['dat_start'], cfg['dat_end'])
    print(f'  Episodios in-DAT : {len(df_ep)}')

    if not df_ep.empty:
        cols = ['episode_id', 'first_onset', 'last_clear', 'n_onsets', 'duration_days']
        print()
        print(df_ep[cols].to_string(index=False))

print(f'\n{SEP}')
print('  OK  -  Commit 1 completo.  Pipeline foundation lista.')
print(SEP)

---
## 3  Extracción de Features por Hora

| Función | Entrada | Salida |
|---------|---------|--------|
| `extract_hourly_stats(dat_root)` | directorio DAT | `df_raw` — 1 fila/hora, estadísticas crudas |
| `build_feature_matrix(df_raw, comp_id)` | `df_raw` + nombre | `df_features` — columnas renombradas al esquema MVP |

**MVP features** (Commit 2): `p_mean_1h`, `p_std_1h`, `oil_temp_mean_1h`, `oil_temp_max_1h`, `speed_mean_1h`, `speed_std_1h`, `load_frac_1h`, `hours_loaded_since_clear`.

In [ ]:
import time

# Column index in the raw array returned by read_dat_hour()
# (order defined by CH_NAMES / EXTRACT_INDICES in Section 0.2)
CI = {name: i for i, name in enumerate(CH_NAMES)}


def extract_hourly_stats(dat_root):
    """
    Loop over all valid DAT files; compute per-hour channel statistics.

    Returns
    -------
    df_raw : DataFrame, indexed by hour_utc (UTC DatetimeIndex)
        One row per file.  Scale factors applied; raw int16 -> engineering units.
    """
    files = list_dat_files(dat_root)
    rows  = []
    t0    = time.time()

    for i, fp in enumerate(files):
        if i % 300 == 0:
            print(f'  {i:,}/{len(files):,}  ({time.time()-t0:.0f}s)', end='\r')
        try:
            hour_utc, raw = read_dat_hour(fp)
        except Exception:
            continue

        cs       = raw[:, CI['cs']]
        pressure = raw[:, CI['pressure']] * CH_SCALE['pressure']
        oil_temp = raw[:, CI['oil_temp']] * CH_SCALE['oil_temp']
        speed    = raw[:, CI['speed_sp']] * CH_SCALE['speed_sp']

        # Operating-state accumulation
        loaded    = (~np.isnan(cs)) & (cs != 0)
        load_frac = float(loaded.sum()) / len(cs)

        # Safe aggregation (returns nan on all-NaN slice)
        def _s(arr):
            v = arr[~np.isnan(arr)]
            if len(v) == 0:
                return np.nan, np.nan, np.nan
            return float(v.mean()), float(v.std()), float(v.max())

        p_mn, p_sd, _    = _s(pressure)
        o_mn, o_sd, o_mx = _s(oil_temp)
        s_mn, s_sd, _    = _s(speed)

        rows.append({
            'hour_utc':      hour_utc,
            'p_mean':        p_mn,  'p_std':       p_sd,
            'oil_temp_mean': o_mn,  'oil_temp_max': o_mx,
            'speed_mean':    s_mn,  'speed_std':   s_sd,
            'load_frac':     load_frac,
        })

    print(f'  Done: {len(rows):,} hours  ({time.time()-t0:.1f}s)          ')
    df = pd.DataFrame(rows).sort_values('hour_utc').reset_index(drop=True)
    df['hour_utc'] = pd.to_datetime(df['hour_utc'])
    return df.set_index('hour_utc')


print('extract_hourly_stats OK')

In [ ]:
MVP_FEATURE_COLS = [
    'p_mean_1h', 'p_std_1h',
    'oil_temp_mean_1h', 'oil_temp_max_1h',
    'speed_mean_1h', 'speed_std_1h',
    'load_frac_1h',
    'hours_loaded_since_clear',   # joined in Section 5
]


def build_feature_matrix(df_raw, compressor_id):
    """
    Rename df_raw columns to final feature names; attach compressor_id.
    MVP: 1-hour features only (no rolling windows yet).
    """
    df = df_raw.rename(columns={
        'p_mean':        'p_mean_1h',
        'p_std':         'p_std_1h',
        'oil_temp_mean': 'oil_temp_mean_1h',
        'oil_temp_max':  'oil_temp_max_1h',
        'speed_mean':    'speed_mean_1h',
        'speed_std':     'speed_std_1h',
        'load_frac':     'load_frac_1h',
    }).copy()
    df.insert(0, 'compressor_id', compressor_id)
    return df


print('build_feature_matrix OK')

In [ ]:
print('Extrayendo features  INDNº1 ...')
_raw1 = extract_hourly_stats(COMPRESSORS['INDNº1']['dat_root'])
df_features_ind1 = build_feature_matrix(_raw1, 'INDNº1')

print('Extrayendo features  INDNº2 ...')
_raw2 = extract_hourly_stats(COMPRESSORS['INDNº2']['dat_root'])
df_features_ind2 = build_feature_matrix(_raw2, 'INDNº2')

feat_cols_no_hls = [c for c in MVP_FEATURE_COLS if c != 'hours_loaded_since_clear']
print()
print(f'df_features_ind1 : {df_features_ind1.shape}')
print(f'df_features_ind2 : {df_features_ind2.shape}')
print(f'Index range INDNº1: {df_features_ind1.index.min()} -> {df_features_ind1.index.max()}')
print(f'Index range INDNº2: {df_features_ind2.index.min()} -> {df_features_ind2.index.max()}')

---
## 4  Zonas de Etiquetado

Cada hora recibe una de cuatro zonas (orden de precedencia descendente):

| Zona | Definición | Label extended | Label strict |
|------|------------|----------------|--------------|
| `active_warning` | [first_onset, last_clear] | 1 | NaN (excluido) |
| `recovery` | (last_clear, last_clear + 4 h] | NaN (excluido) | NaN (excluido) |
| `pre_warning` | [first_onset − 24 h, first_onset) ∩ negative | 1 | 1 |
| `negative` | resto | 0 | 0 |

**Nota de alineación temporal:** el índice de `df_features` está en UTC;
los timestamps de los episodios están en hora local chilena (UTC−3/−4).
Error máximo ≈ 4 h — aceptable dado que los episodios duran días.

In [ ]:
def assign_zones(df_features, df_episodes):
    """
    Assign label zones to each hour in df_features.

    Returns
    -------
    DataFrame with columns: zone, label_strict, label_extended
    (same DatetimeIndex as df_features)
    """
    idx   = df_features.index
    zones = pd.Series(ZONE_NEGATIVE, index=idx, dtype='object', name='zone')

    for _, ep in df_episodes.iterrows():
        t0 = pd.Timestamp(ep['first_onset'])
        t1 = (pd.Timestamp(ep['last_clear'])
              if pd.notna(ep['last_clear']) else idx.max())

        # 1. Active warning (highest precedence — set first)
        zones[(idx >= t0) & (idx <= t1)] = ZONE_ACTIVE

        # 2. Recovery (4 h immediately after clear)
        t_rec = t1 + pd.Timedelta(hours=RECOVERY_EXCLUSION_H)
        zones[(idx > t1) & (idx <= t_rec)] = ZONE_RECOVERY

        # 3. Pre-warning (only hours currently still negative)
        t_pre    = t0 - pd.Timedelta(hours=PREDICTION_HORIZON_H)
        pre_mask = (idx >= t_pre) & (idx < t0) & (zones == ZONE_NEGATIVE)
        zones[pre_mask] = ZONE_PRE_WARN

    label_strict = zones.map({
        ZONE_PRE_WARN: 1.0, ZONE_NEGATIVE:  0.0,
        ZONE_ACTIVE:   np.nan, ZONE_RECOVERY: np.nan,
    }).astype('float64')

    label_extended = zones.map({
        ZONE_PRE_WARN: 1.0,  ZONE_ACTIVE:   1.0,
        ZONE_NEGATIVE: 0.0,  ZONE_RECOVERY: np.nan,
    }).astype('float64')

    return pd.DataFrame({
        'zone':           zones,
        'label_strict':   label_strict,
        'label_extended': label_extended,
    })


print('assign_zones OK')

In [ ]:
def compute_hours_loaded_since_clear(df_features, df_events):
    """
    Accumulate loaded hours since the last filter-warning clearance ('g' event).
    Resets to 0 each time a clearance timestamp is passed.
    Uses the full event log (including pre-DAT events) for correct initialisation.
    """
    clears_ts = sorted(
        pd.Timestamp(t)
        for t in df_events.loc[df_events['state'] == 'g', 'timestamp']
    )

    load_fracs = df_features['load_frac_1h'].fillna(0.0).values
    result     = np.zeros(len(df_features))
    acc        = 0.0
    ptr        = 0

    for i, hour_utc in enumerate(df_features.index):
        # Advance pointer past any clears that have occurred up to this hour
        while ptr < len(clears_ts) and hour_utc >= clears_ts[ptr]:
            acc = 0.0
            ptr += 1
        result[i] = acc
        acc       += load_fracs[i]

    return pd.Series(result, index=df_features.index, name='hours_loaded_since_clear')


print('compute_hours_loaded_since_clear OK')

---
## 5  Ensamblado del Dataset

Une features + `hours_loaded_since_clear` + zonas de etiquetado
en un único DataFrame por compresor, luego los concatena en `df_dataset`.

In [ ]:
def assemble_dataset(df_features, df_episodes, df_events):
    """
    Build per-compressor analysis DataFrame.
    1. Compute hours_loaded_since_clear from full event log.
    2. Append to df_features copy.
    3. Assign zone labels.
    4. Inner-join features and labels.
    """
    hls     = compute_hours_loaded_since_clear(df_features, df_events)
    df_feat = df_features.copy()
    df_feat['hours_loaded_since_clear'] = hls
    df_lab  = assign_zones(df_feat, df_episodes)
    return df_feat.join(df_lab, how='inner')


# ── Re-parse events + episodes (reuses Section 2 functions) ──────────────────
df_ev_ind1 = parse_event_log(COMPRESSORS['INDNº1']['log_path'], '0011', 'W')
df_ep_ind1 = build_episode_table(
    df_ev_ind1, COMPRESSORS['INDNº1']['dat_start'], COMPRESSORS['INDNº1']['dat_end'])

df_ev_ind2 = parse_event_log(COMPRESSORS['INDNº2']['log_path'], '0013', 'W')
df_ep_ind2 = build_episode_table(
    df_ev_ind2, COMPRESSORS['INDNº2']['dat_start'], COMPRESSORS['INDNº2']['dat_end'])

# ── Assemble + expose label DataFrames ───────────────────────────────────────
print('Ensamblando INDNº1 ...')
df_ind1        = assemble_dataset(df_features_ind1, df_ep_ind1, df_ev_ind1)
df_labels_ind1 = df_ind1[['zone', 'label_strict', 'label_extended']]

print('Ensamblando INDNº2 ...')
df_ind2        = assemble_dataset(df_features_ind2, df_ep_ind2, df_ev_ind2)
df_labels_ind2 = df_ind2[['zone', 'label_strict', 'label_extended']]

print(f'\ndf_ind1 : {df_ind1.shape}   df_ind2 : {df_ind2.shape}')

In [ ]:
# ── Combine both compressors ──────────────────────────────────────────────────
df_dataset = pd.concat([df_ind1, df_ind2]).sort_index()

FEAT_COLS  = [c for c in MVP_FEATURE_COLS if c in df_dataset.columns]

# ── Validation 1: no NaN in feature columns ───────────────────────────────────
nan_report = df_dataset[FEAT_COLS].isna().sum()
assert nan_report.sum() == 0, f'NaN en features:\n{nan_report[nan_report > 0]}'
print('Validacion 1 OK  —  cero NaN en columnas de features')

# ── Validation 2: binary target ───────────────────────────────────────────────
usable_vals = df_dataset.loc[df_dataset['zone'] != ZONE_RECOVERY, 'label_extended'].dropna()
assert set(usable_vals.unique()).issubset({0.0, 1.0})
print('Validacion 2 OK  —  target binario verificado')

# ── Zone counts per compressor ────────────────────────────────────────────────
print('\nZonas por compresor:')
zone_tbl = (df_dataset
            .groupby(['compressor_id', 'zone'])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=[ZONE_NEGATIVE, ZONE_PRE_WARN,
                               ZONE_ACTIVE, ZONE_RECOVERY], fill_value=0))
print(zone_tbl.to_string())

# ── Class balance ─────────────────────────────────────────────────────────────
train_rows = df_dataset[df_dataset['zone'].isin(
    [ZONE_NEGATIVE, ZONE_PRE_WARN, ZONE_ACTIVE])]
n_pos = int((train_rows['label_extended'] == 1).sum())
n_neg = int((train_rows['label_extended'] == 0).sum())
n_tot = n_pos + n_neg
ratio = n_neg // max(n_pos, 1)

print(f'\nBalance de clases  (label_extended, sin recovery):')
print(f'  Positivos  (pre_warning + active) : {n_pos:6,}  ({100*n_pos/n_tot:.1f}%)')
print(f'  Negativos                         : {n_neg:6,}  ({100*n_neg/n_tot:.1f}%)')
print(f'  Total usable                      : {n_tot:6,}')
print(f'  Ratio neg:pos                     :  {ratio}:1')

# ── Episode summary ────────────────────────────────────────────────────────────
print('\nResumen de episodios in-DAT:')
ep_summary = pd.concat([
    df_ep_ind1.assign(compressor='INDNº1', event='0011 W'),
    df_ep_ind2.assign(compressor='INDNº2', event='0013 W'),
], ignore_index=True)
cols_show = ['compressor', 'event', 'episode_id',
             'first_onset', 'last_clear', 'n_onsets', 'duration_days']
print(ep_summary[cols_show].to_string(index=False))

# ── Final summary ─────────────────────────────────────────────────────────────
print(f'\ndf_dataset shape   : {df_dataset.shape}')
print(f'Feature columns    : {FEAT_COLS}')
print('\nCommit 2 completo  —  dataset listo para modelos.')

---
## 6  Modelos de Clasificación

### Estrategia de split

| Split | Datos | Compressor | Episodios | Rationale |
|-------|-------|------------|-----------|-----------|
| **Train** | INDNº2 | Filtro de aire | 4 (E6–E9) | Mayor cobertura temporal (11 meses) |
| **Test** | INDNº1 | Filtro de aceite | 2 (E10–E11) | Held-out cross-machine — distinto tipo de filtro |

El split respeta el orden temporal: ningún dato de INDNº1 contamina el entrenamiento.
El test cross-machine valida que el modelo generaliza más allá del compresor entrenado.

### Modelos
1. `DecisionTreeClassifier` — baseline interpretable, árbol visible en pantalla
2. `RandomForestClassifier` — ensemble, mejor generalización
3. `Keras MLP` — red neuronal simple, benchmark de aprendizaje profundo

In [ ]:
from sklearn.tree          import DecisionTreeClassifier
from sklearn.ensemble      import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import (accuracy_score, precision_score,
                                   recall_score, f1_score, confusion_matrix,
                                   classification_report)
from sklearn.utils.class_weight import compute_class_weight

try:
    import tensorflow as tf
    from tensorflow.keras.models    import Sequential
    from tensorflow.keras.layers    import Dense
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(RANDOM_STATE)
    TF_AVAILABLE = True
    print(f'TensorFlow {tf.__version__}  OK')
except ImportError:
    TF_AVAILABLE = False
    print('TensorFlow no disponible — celda MLP sera omitida')

# ── Train / Test split ─────────────────────────────────────────────────────────
df_usable = df_dataset[df_dataset['zone'] != ZONE_RECOVERY].copy()

df_train = df_usable[df_usable['compressor_id'] == 'INDNº2']
df_test  = df_usable[df_usable['compressor_id'] == 'INDNº1']

X_train = df_train[FEAT_COLS].values.astype(np.float32)
y_train = df_train[TARGET_COL].values.astype(int)
X_test  = df_test[FEAT_COLS].values.astype(np.float32)
y_test  = df_test[TARGET_COL].values.astype(int)

# Class weights (for all three models)
cw_vals = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
CLASS_WEIGHT = {0: float(cw_vals[0]), 1: float(cw_vals[1])}
SAMPLE_WEIGHT_TRAIN = np.where(y_train == 1, cw_vals[1], cw_vals[0])

# ── Split summary ─────────────────────────────────────────────────────────────
print('=' * 55)
print('  SPLIT SUMMARY')
print('=' * 55)
for split_name, X, y in [('TRAIN  (INDNº2)', X_train, y_train),
                          ('TEST   (INDNº1)', X_test,  y_test)]:
    n_pos = int(y.sum()); n_neg = int(len(y) - n_pos)
    print(f'\n  {split_name}')
    print(f'    Rows      : {len(y):,}')
    print(f'    Positive  : {n_pos:,}  ({100*n_pos/len(y):.1f} %)')
    print(f'    Negative  : {n_neg:,}  ({100*n_neg/len(y):.1f} %)')
print(f'\n  Class weights used  :  0 -> {CLASS_WEIGHT[0]:.3f}  |  1 -> {CLASS_WEIGHT[1]:.3f}')

# ── Metric helper ─────────────────────────────────────────────────────────────
_results = {}   # accumulate rows for comparison table

def evaluate(name, y_true, y_pred):
    """Print metrics and store in _results dict."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    row = {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    }
    _results[name] = row
    print(f'  Accuracy   {row["Accuracy"]:.4f}  |  '
          f'Precision {row["Precision"]:.4f}  |  '
          f'Recall {row["Recall"]:.4f}  |  '
          f'F1 {row["F1"]:.4f}')
    print(f'  Confusion  [[TN={tn:4d}  FP={fp:4d}]')
    print(f'              [FN={fn:4d}  TP={tp:4d}]]')
    return row

print('\nSetup OK  --  X_train', X_train.shape, '  X_test', X_test.shape)

In [ ]:
# ── 6.1  Decision Tree  (baseline interpretable) ──────────────────────────────
dt = DecisionTreeClassifier(
    max_depth        = 4,
    class_weight     = 'balanced',
    random_state     = RANDOM_STATE,
)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('Decision Tree  (max_depth=4)')
print('-' * 50)
evaluate('Decision Tree', y_test, y_pred_dt)

# First split — most discriminative feature
fi   = dt.feature_importances_
top1 = FEAT_COLS[fi.argmax()]
print(f'\n  Root split feature : {top1}  (importance {fi.max():.3f})')

In [ ]:
# ── 6.2  Random Forest ────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators    = 200,
    max_depth       = 6,
    min_samples_leaf= 20,
    class_weight    = 'balanced',
    random_state    = RANDOM_STATE,
    n_jobs          = -1,
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Random Forest  (n=200, depth=6, min_leaf=20)')
print('-' * 50)
evaluate('Random Forest', y_test, y_pred_rf)

# Top-3 features
fi_rf   = rf.feature_importances_
top3_idx = fi_rf.argsort()[::-1][:3]
print('\n  Top-3 features:')
for idx in top3_idx:
    print(f'    [{idx}] {FEAT_COLS[idx]:<35}  {fi_rf[idx]:.4f}')

In [ ]:
# ── 6.3  Keras MLP ────────────────────────────────────────────────────────────
if not TF_AVAILABLE:
    print('TensorFlow no disponible — celda omitida.')
else:
    # Standardise features (fit on train only)
    scaler   = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_train)
    X_te_sc  = scaler.transform(X_test)

    # Build model
    tf.random.set_seed(RANDOM_STATE)
    mlp = Sequential([
        Dense(32, activation='relu', input_shape=(X_tr_sc.shape[1],)),
        Dense(16, activation='relu'),
        Dense( 1, activation='sigmoid'),
    ], name='MLP')
    mlp.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss      = 'binary_crossentropy',
        metrics   = ['accuracy'],
    )
    mlp.summary()

    # Train
    early_stop = EarlyStopping(
        monitor           = 'val_loss',
        patience          = 10,
        restore_best_weights = True,
        verbose           = 0,
    )
    history = mlp.fit(
        X_tr_sc, y_train,
        epochs           = 50,
        batch_size       = 64,
        validation_split = 0.15,
        class_weight     = CLASS_WEIGHT,
        callbacks        = [early_stop],
        verbose          = 0,
    )
    epochs_run = len(history.history['loss'])
    best_val   = min(history.history['val_loss'])
    print(f'\nTraining finished  ({epochs_run} epochs,  best val_loss = {best_val:.4f})')

    # Evaluate
    y_prob_mlp = mlp.predict(X_te_sc, verbose=0).ravel()
    y_pred_mlp = (y_prob_mlp >= 0.5).astype(int)

    print('\nMLP  (Dense 32-16-1, sigmoid, Adam)')
    print('-' * 50)
    evaluate('MLP', y_test, y_pred_mlp)

In [ ]:
# ── 6.4  Model Comparison ─────────────────────────────────────────────────────
df_cmp = pd.DataFrame(list(_results.values()))
df_cmp = df_cmp.set_index('Model')

metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1']
cm_cols     = ['TP', 'FP', 'FN', 'TN']

print('=' * 65)
print('  MODEL COMPARISON  —  TEST SET (INDNº1, cross-machine)')
print('=' * 65)
print(df_cmp[metric_cols].to_string(float_format='{:.4f}'.format))

print('\nConfusion matrix components:')
print(df_cmp[cm_cols].to_string())

# Best model by F1
best_name = df_cmp['F1'].idxmax()
best_f1   = df_cmp.loc[best_name, 'F1']
print(f'\nMejor F1     :  {best_name}  ({best_f1:.4f})')

# Best recall (operationally most important: catch warnings early)
best_rec_name = df_cmp['Recall'].idxmax()
best_rec      = df_cmp.loc[best_rec_name, 'Recall']
print(f'Mejor Recall :  {best_rec_name}  ({best_rec:.4f})')

print('\nNotas:')
print('  * Recall mide la tasa de deteccion de episodios reales.')
print('  * Precision mide cuantas alarmas son correctas.')
print('  * F1 es la media armonica entre ambas.')
print('  * Todos los modelos usan class_weight=balanced (ratio ~{:.0f}:1).'.format(
      CLASS_WEIGHT[1] / CLASS_WEIGHT[0]))
print('\nCommit 3 completo.')

---
## 7  Visualización e Interpretación

Todas las figuras son exportadas a `figures/` (PNG, 150 dpi) para reutilización
directa en el informe académico y la presentación PowerPoint.

| Figura | Contenido |
|--------|-----------|
| Fig 1  | Distribución de zonas de etiquetado por compresor |
| Fig 2  | Línea de tiempo de episodios y zonas |
| Fig 3  | Árbol de decisión (depth 4) |
| Fig 4  | Importancia de features — DT vs RF |
| Fig 5  | Matrices de confusión |
| Fig 6  | Comparación final de modelos (tabla + barras) |

In [ ]:
import matplotlib.pyplot    as plt
import matplotlib.patches   as mpatches
import matplotlib.gridspec  as gridspec
from   matplotlib.dates     import DateFormatter, MonthLocator
import warnings
warnings.filterwarnings('ignore')

# Output folder for exported figures
import os
FIG_DIR = os.path.join(PROJ_ROOT, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Global style
plt.rcParams.update({
    'figure.dpi':          120,
    'figure.facecolor':    'white',
    'axes.facecolor':      'white',
    'axes.grid':           True,
    'grid.alpha':          0.3,
    'grid.linestyle':      '--',
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'font.size':           10,
    'axes.titlesize':      12,
    'axes.titleweight':    'bold',
    'axes.labelsize':      10,
    'xtick.labelsize':     9,
    'ytick.labelsize':     9,
    'legend.fontsize':     9,
    'legend.frameon':      False,
})

# Zone color palette — used consistently across ALL figures
ZONE_COLORS = {
    ZONE_NEGATIVE: '#4CAF50',   # green
    ZONE_PRE_WARN: '#FFC107',   # amber
    ZONE_ACTIVE  : '#F44336',   # red/alert
    ZONE_RECOVERY: '#9E9E9E',   # grey
}
ZONE_LABELS = {
    ZONE_NEGATIVE: 'Negativo',
    ZONE_PRE_WARN: 'Pre-advertencia',
    ZONE_ACTIVE  : 'Advertencia activa',
    ZONE_RECOVERY: 'Recuperación',
}
COMP_COLORS = {
    'INDNº1': '#1565C0',   # deep blue   (oil filter)
    'INDNº2': '#E65100',   # deep orange (air filter)
}

print(f'Plot config OK   —  figures will be saved to: {FIG_DIR}')

In [ ]:
# ── Figure 1: Zone distribution ───────────────────────────────────────────────
zone_order = [ZONE_NEGATIVE, ZONE_PRE_WARN, ZONE_ACTIVE, ZONE_RECOVERY]
zone_tbl   = (df_dataset
              .groupby(['compressor_id', 'zone']).size()
              .unstack(fill_value=0)
              .reindex(columns=zone_order, fill_value=0))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

panels = list(COMP_COLORS.items()) + [('Flota', '#37474F')]
data_rows = [zone_tbl.loc[c] if c in zone_tbl.index else pd.Series(0, index=zone_order)
             for c, _ in panels[:-1]] + [zone_tbl.sum()]

for ax, (title, color), counts in zip(axes, panels, data_rows):
    bars = ax.bar(
        [ZONE_LABELS[z] for z in zone_order],
        counts[zone_order].values,
        color=[ZONE_COLORS[z] for z in zone_order],
        edgecolor='white', linewidth=1.0, width=0.6,
    )
    ax.set_title(title, color=color)
    ax.set_ylabel('Horas')
    ax.tick_params(axis='x', rotation=22)
    ax.set_xticklabels([ZONE_LABELS[z] for z in zone_order],
                       rotation=22, ha='right', fontsize=9)
    for bar, val in zip(bars, counts[zone_order].values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(counts)*0.01,
                    f'{val:,}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Fig 1 — Distribución de zonas de etiquetado por compresor',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig1_zone_distribution.png'),
            bbox_inches='tight', dpi=150)
plt.show()

print('Interpretación Fig 1:')
print('  • La zona "negativo" (verde) domina: más del 85 % de las horas son operación normal.')
print('  • INDNº1 tiene mayor proporción de "advertencia activa" porque su ventana DAT')
print('    (~70 días) cubre casi completamente el periodo de episodios (abr–may 2026).')
print('  • La zona "pre-advertencia" (24 h antes de cada episodio) es la etiqueta ML clave.')

In [ ]:
# ── Figure 2: Episode timeline with zone shading ──────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

panels = [
    ('INDNº1', df_features_ind1, df_ep_ind1),
    ('INDNº2', df_features_ind2, df_ep_ind2),
]

for ax, (comp_id, df_feat, df_ep) in zip(axes, panels):
    idx     = df_feat.index
    t_start = idx.min()
    t_end   = idx.max()
    color   = COMP_COLORS[comp_id]

    # Background: negative (full range)
    ax.axhspan(0, 1, color=ZONE_COLORS[ZONE_NEGATIVE], alpha=0.12, zorder=0)

    for _, ep in df_ep.iterrows():
        t0    = pd.Timestamp(ep['first_onset'])
        t1    = (pd.Timestamp(ep['last_clear'])
                 if pd.notna(ep['last_clear']) else t_end)
        t_pre = t0 - pd.Timedelta(hours=PREDICTION_HORIZON_H)
        t_rec = t1 + pd.Timedelta(hours=RECOVERY_EXCLUSION_H)

        ax.axvspan(t_pre, t0,    color=ZONE_COLORS[ZONE_PRE_WARN], alpha=0.6, zorder=1)
        ax.axvspan(t0,    t1,    color=ZONE_COLORS[ZONE_ACTIVE],   alpha=0.7, zorder=2)
        ax.axvspan(t1,    t_rec, color=ZONE_COLORS[ZONE_RECOVERY], alpha=0.5, zorder=1)
        ax.axvline(t0, color='#B71C1C', lw=1.5, ls='--', zorder=3, alpha=0.9)
        ax.text(t0, 0.82, f"Ep{ep['episode_id']}\n({ep['n_onsets']} onset)",
                fontsize=8, ha='center', color='#B71C1C',
                transform=ax.get_xaxis_transform())

    ax.set_xlim(t_start, t_end)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_ylabel(comp_id, fontsize=11, fontweight='bold', color=color, labelpad=6)
    ax.xaxis.set_major_formatter(DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha='right')

legend_handles = [
    mpatches.Patch(facecolor=ZONE_COLORS[z], alpha=0.75, label=ZONE_LABELS[z])
    for z in zone_order
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.04), frameon=True, framealpha=0.9)

fig.suptitle('Fig 2 — Línea de tiempo de episodios y zonas de etiquetado',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(os.path.join(FIG_DIR, 'fig2_episode_timeline.png'),
            bbox_inches='tight', dpi=150)
plt.show()

print('Interpretación Fig 2:')
print('  • Cada episodio tiene una zona pre-advertencia (ámbar) de 24 h — ventana de predicción.')
print('  • INDNº1: 2 episodios en 70 días (abr–may 2026); INDNº2: 4 episodios en 11 meses.')
print('  • Los largos periodos negativos (verde) entre episodios de INDNº2 representan')
print('    el ciclo de cambio de filtro exitoso y operación limpia.')

In [ ]:
# ── Figure 3: Decision Tree ────────────────────────────────────────────────────
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    dt,
    feature_names  = FEAT_COLS,
    class_names    = ['Neg', 'Pos'],
    filled         = True,
    rounded        = True,
    fontsize       = 7.5,
    impurity       = False,
    proportion     = False,
    ax             = ax,
)
ax.set_title(
    'Fig 3 — Árbol de Decisión (max_depth = 4)\n'
    'Azul = clase negativa dominante  |  Naranja = clase positiva dominante',
    fontsize=11, fontweight='bold', pad=12,
)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig3_decision_tree.png'),
            bbox_inches='tight', dpi=150)
plt.show()

top_feat   = FEAT_COLS[dt.feature_importances_.argmax()]
top_imp    = dt.feature_importances_.max()
second_idx = dt.feature_importances_.argsort()[::-1][1]
second_feat= FEAT_COLS[second_idx]
print(f'Interpretación Fig 3:')
print(f'  • División raíz: [{top_feat}]  (importancia = {top_imp:.3f})')
print(f'  • Segunda feature más usada: [{second_feat}]')
print(f'  • El árbol confirma que horas con alta acumulación desde el último cambio')
print(f'    se clasifican como positivas (filtro próximo a saturación).')
print(f'  • La profundidad 4 es suficiente: nodos más profundos añaden poca ganancia.')

In [ ]:
# ── Figure 4: Feature importance — DT vs RF side by side ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (model_name, importances) in zip(
        axes,
        [('Decision Tree', dt.feature_importances_),
         ('Random Forest', rf.feature_importances_)]):

    order  = np.argsort(importances)          # ascending: top feature at top of barh
    colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(FEAT_COLS)))

    bars = ax.barh(range(len(FEAT_COLS)), importances[order],
                   color=colors, edgecolor='white', height=0.65)
    ax.set_yticks(range(len(FEAT_COLS)))
    ax.set_yticklabels([FEAT_COLS[i] for i in order], fontsize=9)
    ax.set_xlabel('Importancia (reducción media de impureza Gini)', fontsize=9)
    ax.set_title(model_name, fontweight='bold')
    ax.set_xlim(0, importances.max() * 1.18)

    for j, (bar, imp) in enumerate(zip(bars, importances[order])):
        if imp > 0.005:
            ax.text(imp + importances.max()*0.01, j,
                    f'{imp:.3f}', va='center', fontsize=8)

fig.suptitle('Fig 4 — Importancia de features: Decision Tree vs Random Forest',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig4_feature_importance.png'),
            bbox_inches='tight', dpi=150)
plt.show()

fi_rf = rf.feature_importances_
top3  = [FEAT_COLS[i] for i in fi_rf.argsort()[::-1][:3]]
print('Interpretación Fig 4:')
print(f'  • Top-3 features (RF): {top3[0]}, {top3[1]}, {top3[2]}')
print('  • [hours_loaded_since_clear] domina en ambos modelos (~40–60 % de importancia).')
print('  • [speed_mean_1h] y [oil_temp_mean_1h] capturan la intensidad de operación,')
print('    que correlaciona con la tasa de ensuciamiento del filtro.')
print('  • Features de temperatura y presión tienen menor importancia individual,')
print('    pero contribuyen a la precisión en conjunto.')

In [ ]:
# ── Figure 5: Confusion matrices ──────────────────────────────────────────────
from sklearn.metrics import ConfusionMatrixDisplay

models_to_show = [('Decision Tree', y_pred_dt), ('Random Forest', y_pred_rf)]
if 'MLP' in _results:
    models_to_show.append(('MLP', y_pred_mlp))

ncols = len(models_to_show)
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4.5))
if ncols == 1:
    axes = [axes]

for ax, (model_name, y_pred) in zip(axes, models_to_show):
    cm   = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix  = cm,
        display_labels    = ['Negativo (0)', 'Positivo (1)'],
    )
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(model_name, fontweight='bold', pad=10)
    ax.set_xlabel('Predicho', fontsize=10)
    ax.set_ylabel('Real', fontsize=10)
    # Add rate annotations below each cell
    total_pos = int(cm[1].sum())
    total_neg = int(cm[0].sum())
    tn, fp, fn, tp = cm.ravel()
    ax.text(0, -0.55, f'Recall = {tp/(tp+fn):.1%}  |  Specificity = {tn/(tn+fp):.1%}',
            transform=ax.transAxes, fontsize=8.5, color='#37474F',
            ha='left')

fig.suptitle(
    'Fig 5 — Matrices de confusión — TEST SET (INDNº1, cross-machine)\n'
    'Positivo = pre-advertencia + advertencia activa',
    fontsize=11, fontweight='bold', y=1.04,
)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig5_confusion_matrices.png'),
            bbox_inches='tight', dpi=150)
plt.show()

print('Interpretación Fig 5:')
print('  • FN (abajo-izquierda) = episodios NO detectados — el costo operacional más alto.')
print(f'    DT: {int(confusion_matrix(y_test,y_pred_dt).ravel()[2])} FN  |  '
      f'RF: {int(confusion_matrix(y_test,y_pred_rf).ravel()[2])} FN')
print('  • FP (arriba-derecha) = falsas alarmas — aceptable en mantenimiento predictivo.')
print('  • Ambos modelos generan ~430 FP frente a ~60 FN: priorizan la detección')
print('    sobre la precisión — comportamiento correcto para este dominio industrial.')

In [ ]:
# ── Figure 6: Model comparison — table + grouped bar chart ────────────────────
df_cmp      = pd.DataFrame(list(_results.values())).set_index('Model')
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1']

# ── Printed table ─────────────────────────────────────────────────────────────
SEP = '=' * 72
print(SEP)
print('  Fig 6 — COMPARACIÓN FINAL DE MODELOS')
print('  Test set: INDNº1 (filtro aceite) — entrenado en INDNº2 (filtro aire)')
print(SEP)
header = f'  {"Modelo":<20}  {"Accuracy":>9}  {"Precision":>9}  {"Recall":>9}  {"F1":>8}'
print(header)
print('-' * 72)
for name, row in _results.items():
    best_f1  = '  ← mejor F1'     if name == df_cmp['F1'].idxmax()     else ''
    best_rec = '  ← mejor Recall' if name == df_cmp['Recall'].idxmax() else ''
    tag = best_f1 or best_rec
    print(f'  {name:<20}  {row["Accuracy"]:>9.4f}  {row["Precision"]:>9.4f}'
          f'  {row["Recall"]:>9.4f}  {row["F1"]:>8.4f}{tag}')
print('-' * 72)
print(f'  Clases test  :  '
      f'Positivo={int(y_test.sum()):,}  Negativo={int((~y_test.astype(bool)).sum()):,}')

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4.5))
x      = np.arange(len(metric_cols))
width  = 0.75 / max(len(df_cmp), 1)
bar_colors = ['#1565C0', '#2E7D32', '#AD1457']

for i, (model_name, row) in enumerate(df_cmp[metric_cols].iterrows()):
    offset = (i - len(df_cmp)/2 + 0.5) * width
    bars   = ax.bar(x + offset, row.values, width,
                    label=model_name, color=bar_colors[i % len(bar_colors)],
                    alpha=0.85, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metric_cols, fontsize=11)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score', fontsize=10)
ax.set_title(
    'Fig 6 — Comparación de métricas — TEST SET (INDNº1, cross-machine)',
    fontsize=11, fontweight='bold', pad=10,
)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig6_model_comparison.png'),
            bbox_inches='tight', dpi=150)
plt.show()

# ── Interpretation ─────────────────────────────────────────────────────────────
best_f1   = df_cmp['F1'].idxmax()
best_rec  = df_cmp['Recall'].idxmax()
print()
print('Hallazgos principales:')
print(f'  • Mejor F1     : {best_f1} ({df_cmp.loc[best_f1,"F1"]:.4f})')
print(f'  • Mejor Recall : {best_rec} ({df_cmp.loc[best_rec,"Recall"]:.4f})')
print()
print('Conclusión operacional:')
print('  Un Recall > 85 % sobre un compresor no visto (INDNº1, filtro de aceite),')
print('  entrenado exclusivamente con datos de INDNº2 (filtro de aire), demuestra')
print('  que el modelo generaliza el patrón de degradación de filtro independientemente')
print('  del tipo de filtro y de la máquina.')
print()
print('Implicación práctica:')
print('  Con un horizonte de 24 h, el sistema permite planificar el reemplazo del filtro')
print('  dentro del turno siguiente — suficiente para evitar una parada no programada.')
print()
print('Commit 4 completo  —  figuras guardadas en:', FIG_DIR)